## AND-103 Task 5: Feature Engineering Pipeline

Builds a feature matrix from `inspection.csv`, `order.csv`, and `merged_elevator_data.csv`.
Produces `data/feature_matrix.csv` consumed by Task 6 (ML pipeline).

**Leakage constraint:** for any inspection row with date D, every feature value is derived
exclusively from inspections and orders dated strictly before D.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## AND-103 Task 5: Step 1 — Load and clean inspection.csv

In [2]:
insp = pd.read_csv('../data/inspection.csv', parse_dates=['Latest_INSPECTION_Date'])
print(f'Loaded {len(insp):,} inspection rows')
print(f'Date range: {insp["Latest_INSPECTION_Date"].min().date()} → {insp["Latest_INSPECTION_Date"].max().date()}')

Loaded 143,181 inspection rows
Date range: 2011-01-04 → 2017-01-09


In [3]:
# Map raw InspectionOutcome to four model classes.
# Spec groups 20+ raw values into Pass / Follow Up / Fail / Other.
OUTCOME_MAP = {
    'Passed':               'Pass',
    'Passed Major':         'Pass',
    'Passed Sub':           'Pass',
    'All Orders Resolved':  'Pass',
    'Complete':             'Pass',
    'Follow up':            'Follow Up',
    'DC Follow up':         'Follow Up',
    'Follow up Major':      'Follow Up',
    'Follow up Sub Major':  'Follow Up',
    'Follow Up Initial':    'Follow Up',
    'DC Follow up Intial':  'Follow Up',
    'MCP DC Follow up':     'Follow Up',
    'Follow up Sub':        'Follow Up',
    'Shutdown':             'Fail',
    'Vol Shut Down':        'Fail',
    'Fail Initial':         'Fail',
    'Fail Sub':             'Fail',
}
insp['outcome_class'] = insp['InspectionOutcome'].map(OUTCOME_MAP).fillna('Other')

dist = insp['outcome_class'].value_counts()
print('Outcome class distribution:')
print(dist.to_string())
print(f'\nRows mapped to Other: {(insp["outcome_class"] == "Other").sum():,}')
print('Other rows are retained; the model will learn to predict or ignore this class.')

Outcome class distribution:
outcome_class
Follow Up    80574
Pass         53891
Fail          7212
Other         1504

Rows mapped to Other: 1,504
Other rows are retained; the model will learn to predict or ignore this class.


## AND-103 Task 5: Step 2 — Build prior inspection features

**Algorithm:** aggregate to one row per (elevator, date), then apply cumulative sum + shift(1)
within each elevator group. `shift(1)` moves the cumulative total one date backwards, so each
row sees only the sum of dates strictly before its own — the current date is never included.
This is correct even when multiple inspections share the same date.

In [4]:
# One row per (elevator, date) — the unit of aggregation for leakage-safe cumulation.
daily = (
    insp
    .groupby(['ElevatingDevicesNumber', 'Latest_INSPECTION_Date'])
    .agg(
        day_pass     = ('outcome_class', lambda x: (x == 'Pass').sum()),
        day_followup = ('outcome_class', lambda x: (x == 'Follow Up').sum()),
        day_fail     = ('outcome_class', lambda x: (x == 'Fail').sum()),
        day_count    = ('InspectionNumber', 'count'),
        day_last_outcome = ('outcome_class', 'last'),
    )
    .reset_index()
    .sort_values(['ElevatingDevicesNumber', 'Latest_INSPECTION_Date'])
)
print(f'Daily inspection records: {len(daily):,}')

Daily inspection records: 132,212


In [5]:
# Prior outcome counts: cumsum of all previous dates, then shift so current date is excluded.
for col, day_col in [
    ('prior_pass_count',     'day_pass'),
    ('prior_followup_count', 'day_followup'),
    ('prior_fail_count',     'day_fail'),
    ('prior_total_count',    'day_count'),
]:
    daily[col] = (
        daily.groupby('ElevatingDevicesNumber')[day_col]
             .transform(lambda x: x.cumsum().shift(1))
    )

In [6]:
# Days since most recent prior inspection.
# Kept as NaN for an elevator's first inspection — filling with 0 would falsely imply
# the elevator was inspected that same day.
daily['prev_date'] = (
    daily.groupby('ElevatingDevicesNumber')['Latest_INSPECTION_Date'].shift(1)
)
daily['days_since_last_inspection'] = (
    (daily['Latest_INSPECTION_Date'] - daily['prev_date']).dt.days
)

In [7]:
# Most recent prior outcome class.
daily['most_recent_prior_outcome'] = (
    daily.groupby('ElevatingDevicesNumber')['day_last_outcome'].shift(1)
)

# Rolling pass rate: passes / total prior inspections (expanding window).
# An expanding window is used rather than a fixed window (e.g. last 5 dates) because
# the median elevator has only ~3 inspections in the dataset — a fixed window would
# produce NaN for most rows and discard useful signal.
daily['rolling_pass_rate'] = daily['prior_pass_count'] / daily['prior_total_count']

In [8]:
# Keep only the derived feature columns for the merge back to individual inspection rows.
daily_features = daily[[
    'ElevatingDevicesNumber', 'Latest_INSPECTION_Date',
    'prior_pass_count', 'prior_followup_count', 'prior_fail_count',
    'days_since_last_inspection', 'most_recent_prior_outcome', 'rolling_pass_rate',
]]

# Many-to-one merge: multiple inspection rows per (elevator, date) all receive the same
# prior-history features computed at the daily level.
before = len(insp)
insp = insp.merge(daily_features, on=['ElevatingDevicesNumber', 'Latest_INSPECTION_Date'], how='left')
print(f'Rows before merge: {before:,}  |  after: {len(insp):,}')

# Quick sanity check against manually verified values (test_features.py assertions).
row = insp[
    (insp['ElevatingDevicesNumber'] == 17489) &
    (insp['Latest_INSPECTION_Date'] == '2014-02-12')
].iloc[0]
print(f'\nSanity check — elevator 17489 @ 2014-02-12:')
print(f'  prior_pass={row.prior_pass_count}  prior_followup={row.prior_followup_count}  prior_fail={row.prior_fail_count}')
print(f'  Expected: pass=2  followup=3  fail=4')

Rows before merge: 143,181  |  after: 143,181

Sanity check — elevator 17489 @ 2014-02-12:
  prior_pass=2.0  prior_followup=3.0  prior_fail=4.0
  Expected: pass=2  followup=3  fail=4


## AND-103 Task 5: Step 3 — Build prior order features

The spec requires filtering inspections by date first, then using the resulting
inspection numbers to select orders. The daily-aggregate + cumsum approach implements
this correctly: orders are tagged with their inspection's date via `inspectionnumber`,
aggregated by date, then shifted — so each inspection row sees only orders from
inspections with dates strictly before D.

In [9]:
orders = pd.read_csv('../data/order.csv')
DROP_COLS = [
    'DIRECTIVE', 'ClauseText', 'RegulationReference', 'ClauseNumber',
    'TSSAStandardOrderNumber', 'Inspectionsadditionalinformation',
]
orders = orders.drop(columns=DROP_COLS, errors='ignore')
print(f'Loaded {len(orders):,} order rows')
print(f'Remaining columns: {list(orders.columns)}')

Loaded 162,172 order rows
Remaining columns: ['ElevatingDevicesNumber', 'RISKSCORE', 'Inspection_type', 'DateofIssue', 'StatusofInspectionOrder', 'inspectionnumber', 'DaystoComply', 'ComplianceDate', 'customerorderedtocomply']


In [10]:
# RISKSCORE analysis.
null_count = orders['RISKSCORE'].isna().sum()
print(f'RISKSCORE nulls: {null_count:,} of {len(orders):,} ({null_count/len(orders)*100:.1f}%)')
print(orders['RISKSCORE'].describe())
print()
print('The distribution is heavily right-skewed (median 15, max > 20,000).')
print('Strategy: impute nulls with the global median before aggregating.')
print('The median is robust to extreme outliers where the mean would be inflated.')

RISKSCORE nulls: 41,553 of 162,172 (25.6%)
count    120619.000000
mean         11.520786
std         190.868164
min           0.000000
25%           0.000015
50%          15.000000
75%          15.000000
max       20316.561950
Name: RISKSCORE, dtype: float64

The distribution is heavily right-skewed (median 15, max > 20,000).
Strategy: impute nulls with the global median before aggregating.
The median is robust to extreme outliers where the mean would be inflated.


In [11]:
global_median_risk = orders['RISKSCORE'].median()
print(f'Global RISKSCORE median: {global_median_risk}')
orders['RISKSCORE_imputed'] = orders['RISKSCORE'].fillna(global_median_risk)

Global RISKSCORE median: 15.0


In [12]:
# Tag each order with the date of its associated inspection.
# orders.ElevatingDevicesNumber already exists; we only need the date from insp.
date_map = insp[['InspectionNumber', 'Latest_INSPECTION_Date']].drop_duplicates()
orders_dated = orders.merge(
    date_map, left_on='inspectionnumber', right_on='InspectionNumber', how='left'
)
unmatched = orders_dated['Latest_INSPECTION_Date'].isna().sum()
print(f'Orders with matched inspection date: {orders_dated["Latest_INSPECTION_Date"].notna().sum():,}')
print(f'Orders without match (excluded): {unmatched:,}')
orders_dated = orders_dated.dropna(subset=['Latest_INSPECTION_Date'])

Orders with matched inspection date: 162,172
Orders without match (excluded): 0


In [13]:
# Daily order aggregate per (elevator, inspection_date).
daily_orders = (
    orders_dated
    .groupby(['ElevatingDevicesNumber', 'Latest_INSPECTION_Date'])
    .agg(
        day_order_count = ('inspectionnumber', 'count'),
        day_risk_sum    = ('RISKSCORE_imputed', 'sum'),
    )
    .reset_index()
    .sort_values(['ElevatingDevicesNumber', 'Latest_INSPECTION_Date'])
)

# Cumulative order count (prior dates only).
daily_orders['prior_order_count'] = (
    daily_orders.groupby('ElevatingDevicesNumber')['day_order_count']
                .transform(lambda x: x.cumsum().shift(1))
                .fillna(0)
                .astype(int)
)

# Cumulative risk sum, then divide by count for a proper weighted mean.
# (Averaging per-day means would distort days with many orders vs. few.)
daily_orders['prior_risk_sum'] = (
    daily_orders.groupby('ElevatingDevicesNumber')['day_risk_sum']
                .transform(lambda x: x.cumsum().shift(1))
)
daily_orders['prior_mean_riskscore'] = (
    daily_orders['prior_risk_sum'] / daily_orders['prior_order_count']
)

print(f'Daily order records: {len(daily_orders):,}')

Daily order records: 45,503


In [14]:
# Merge order features back onto the inspection rows.
order_features = daily_orders[[
    'ElevatingDevicesNumber', 'Latest_INSPECTION_Date',
    'prior_order_count', 'prior_mean_riskscore',
]]
before = len(insp)
insp = insp.merge(order_features, on=['ElevatingDevicesNumber', 'Latest_INSPECTION_Date'], how='left')
insp['prior_order_count'] = insp['prior_order_count'].fillna(0).astype(int)
print(f'Rows before merge: {before:,}  |  after: {len(insp):,}')

# Sanity check against manually verified value (test_features.py Test 3).
row = insp[
    (insp['ElevatingDevicesNumber'] == 17489) &
    (insp['Latest_INSPECTION_Date'] == '2014-02-12')
].iloc[0]
print(f'\nSanity check — elevator 17489 @ 2014-02-12:')
print(f'  prior_order_count={row.prior_order_count}  (expected 15)')

Rows before merge: 143,181  |  after: 143,181

Sanity check — elevator 17489 @ 2014-02-12:
  prior_order_count=15  (expected 15)


## AND-103 Task 5: Step 4 — Join static features

In [15]:
# Static features from AND-102 Task 5 ETL output.
# Device Type is already cleaned (Freight Elevator variants merged, rare types → Other).
# alteration_count is a single integer per elevator.
# These carry no leakage risk as they do not vary over time.
static = pd.read_csv(
    '../data/merged_elevator_data.csv',
    usecols=['ElevatingDevicesNumber', 'Device Type', 'alteration_count'],
)
before = len(insp)
insp = insp.merge(static, on='ElevatingDevicesNumber', how='left')
print(f'Rows before join: {before:,}  |  after: {len(insp):,}')
print(f'Device Type categories: {insp["Device Type"].value_counts().to_dict()}')

Rows before join: 143,181  |  after: 143,181
Device Type categories: {'Passenger Elevator': 131227, 'Freight Elevator': 5621, 'LULA Elevator': 2461, 'Observation Elevator': 988, 'Other': 21}


## AND-103 Task 5: Step 5 — Encode categorical variables

Two categorical columns are encoded as dummy variables:

1. **`Device Type`** — 7 cleaned categories from AND-102 Task 5.
2. **`most_recent_prior_outcome`** — 4 outcome classes + NaN (first inspection).
   NaN gets its own indicator column (`dummy_na=True`) to distinguish
   "no prior history" from any actual outcome class.

**`InspectionType` (current) is excluded.** The spec explicitly lists it as
a leakage source: the type of an inspection cannot be known before the inspection occurs.
Its prior value is already captured indirectly via `most_recent_prior_outcome`.

In [16]:
before_cols = len(insp.columns)
insp = pd.get_dummies(
    insp,
    columns=['Device Type', 'most_recent_prior_outcome'],
    prefix=['device_type', 'prior_outcome'],
    dummy_na=True,
)
print(f'Columns before encoding: {before_cols}  |  after: {len(insp.columns)}')
dummy_cols = [c for c in insp.columns if c.startswith(('device_type_', 'prior_outcome_'))]
print(f'New dummy columns: {dummy_cols}')

Columns before encoding: 20  |  after: 29
New dummy columns: ['device_type_Freight Elevator', 'device_type_LULA Elevator', 'device_type_Observation Elevator', 'device_type_Other', 'device_type_Passenger Elevator', 'device_type_nan', 'prior_outcome_Fail', 'prior_outcome_Follow Up', 'prior_outcome_Other', 'prior_outcome_Pass', 'prior_outcome_nan']


## AND-103 Task 5: Step 6 — Handle missing values

In [17]:
print('NaN counts before fill:')
nan_counts = insp.isna().sum()
print(nan_counts[nan_counts > 0].to_string())

NaN counts before fill:
InspectionLocation               113
prior_pass_count               43324
prior_followup_count           43324
prior_fail_count               43324
days_since_last_inspection     43324
rolling_pass_rate              43324
prior_mean_riskscore          118957
alteration_count               53179


In [18]:
# Prior inspection counts: NaN means first-ever inspection — no prior history → 0.
for col in ['prior_pass_count', 'prior_followup_count', 'prior_fail_count']:
    insp[col] = insp[col].fillna(0).astype(int)

# Rolling pass rate: NaN on first inspection → 0 (no prior passes to rate).
insp['rolling_pass_rate'] = insp['rolling_pass_rate'].fillna(0)

# days_since_last_inspection: intentionally kept as NaN for first inspections.
# Filling with 0 would imply the elevator was inspected the same day, which is wrong.
# Tree-based models handle NaN natively; other models must impute in the ML pipeline.

# prior_mean_riskscore: NaN when no prior orders exist → fill with global median.
riskscore_fill = global_median_risk
insp['prior_mean_riskscore'] = insp['prior_mean_riskscore'].fillna(riskscore_fill)

# alteration_count: NaN means no alteration records found in AND-102 ETL → 0.
insp['alteration_count'] = insp['alteration_count'].fillna(0).astype(int)

print('NaN counts after fill:')
nan_counts = insp.isna().sum()
remaining = nan_counts[nan_counts > 0]
print(remaining.to_string() if len(remaining) else 'None')
print()
print('Only days_since_last_inspection should remain NaN (by design — first inspections).')

NaN counts after fill:
InspectionLocation              113
days_since_last_inspection    43324

Only days_since_last_inspection should remain NaN (by design — first inspections).


## AND-103 Task 5: Step 7 — Save feature matrix

In [19]:
# Drop columns that are targets, identifiers, or leakage sources.
DROP_FROM_MATRIX = [
    'InspectionOutcome',                  # raw outcome — replaced by outcome_class (target)
    'InspectionType',                     # current inspection type — leakage
    'originatingservicerequestnumber',    # admin identifier, not a feature
    'InspectionCustomer',                 # admin identifier, not a feature
    'InspectionLocation',                 # free-text location, not in scope
    'Earliest_INSPECTION_Date',           # not used in this pipeline
    'InspectionNumber',                   # inspection-level identifier, not a feature
]
feature_matrix = insp.drop(columns=[c for c in DROP_FROM_MATRIX if c in insp.columns])

# Latest_INSPECTION_Date is retained — required for the time-based train/test split in Task 6.
# outcome_class is retained as the target variable column.

feature_matrix.to_csv('../data/feature_matrix.csv', index=False)
print(f'Saved: {feature_matrix.shape[0]:,} rows x {feature_matrix.shape[1]} columns')
print(f'Columns ({feature_matrix.shape[1]}):')
print(list(feature_matrix.columns))

Saved: 143,181 rows x 22 columns
Columns (22):
['ElevatingDevicesNumber', 'Latest_INSPECTION_Date', 'outcome_class', 'prior_pass_count', 'prior_followup_count', 'prior_fail_count', 'days_since_last_inspection', 'rolling_pass_rate', 'prior_order_count', 'prior_mean_riskscore', 'alteration_count', 'device_type_Freight Elevator', 'device_type_LULA Elevator', 'device_type_Observation Elevator', 'device_type_Other', 'device_type_Passenger Elevator', 'device_type_nan', 'prior_outcome_Fail', 'prior_outcome_Follow Up', 'prior_outcome_Other', 'prior_outcome_Pass', 'prior_outcome_nan']
